# Supplementary Figure S1 — Diffusion is not identifiable at 48-h sampling (S10)

Sources: `benchmarks/synthetic_multimodal/s10/diffusion.csv` (per seed × embedding × estimator × variant:
median estimate, median truth trace(D_eff)/d, ratio, Spearman), `diffusion_summary.json`
(measurement-noise reference Σ_meas per unit). v3_default seeds 0–2.

In [1]:
from _style import *
import json as _json

prov = Provenance("FigS1", notebook="notebooks/supp_s10_diffusion.ipynb", kind="supplementary")
df = pd.read_csv(prov.use("benchmarks/synthetic_multimodal/s10/diffusion.csv", "a,c", "S10"))
summ = _json.loads(prov.use("benchmarks/synthetic_multimodal/s10/diffusion_summary.json", "b", "S10").read_text())
V = prov.value
d = df[(df.preset == "v3_default")]

fig, axs = plt.subplots(1, 3, figsize=(DOUBLE_COL, 2.6), gridspec_kw={"width_ratios": [1.6, 0.9, 1.1], "wspace": 0.55})

# a — estimate / truth ratio
ax = axs[0]
rows = [("clone_dispersion", "literal", "clone dispersion\n(− Σ_meas; gate)"), ("clone_dispersion", "uncorrected", "clone dispersion\nuncorrected"),
        ("ot_noise", "minus_trace_sigma_meas_over_d", "OT noise\n(− Σ_meas; gate)"), ("ot_noise", "uncorrected", "OT noise\nuncorrected"),
        ("score_flow", "raw_shared", "score–flow\nraw labelling"), ("score_flow", "smoothed_shared", "score–flow\nsmoothed")]
for j, (est, var, lab) in enumerate(rows):
    for meth, col, dx in [("pca", C_FLOW, -0.12), ("fa", C_BASE, 0.12)]:
        h = d[(d.estimator == est) & (d.variant == var) & (d.method == meth) & (d.quantity == "trace/d")].sort_values("seed")
        ax.scatter(j + dx + np.array([-0.04, 0, 0.04]), h.ratio_to_truth_all, color=col, s=10, zorder=3,
                   label=meth.upper() if j == 0 else None)
        V(f"a_{est}_{var}_{meth}_ratio_min", h.ratio_to_truth_all.min()); V(f"a_{est}_{var}_{meth}_ratio_max", h.ratio_to_truth_all.max())
ax.axhspan(0.5, 2, color="#E5F1E5", lw=0)
ax.axhline(1, color="black", lw=0.6); ax.axhline(0, color=C_BASE, lw=0.4)
ax.set_xticks(range(len(rows))); ax.set_xticklabels([r[2] for r in rows], rotation=45, ha="right", fontsize=5.5)
ax.set_ylabel("median estimate / median truth\n(trace D_eff / d)"); ax.set_ylim(-8, 4)
ax.legend(fontsize=6, loc="upper left", borderaxespad=0.2); ax.text(len(rows) - 0.45, -0.25, "within ×2", fontsize=6, ha="right", va="top", color="#2E7D32")
panel_letter(ax, "a", x=-0.25)
prov.panel("a", "Diffusion estimate / truth ratio per estimator, v3_default seeds 0-2, PCA and FA; band = within factor 2 (gate)")

# b — measurement noise vs 48-h truth
ax = axs[1]
for i, meth in enumerate(["pca", "fa"]):
    sm = [summ["units"][f"v3_default_s{s}_{meth}"]["sigma_meas_median_trace_over_d"] for s in range(3)]
    tr = [d[(d.seed == s) & (d.method == meth) & (d.estimator == "clone_dispersion") & (d.variant == "uncorrected")].median_truth_all.iloc[0] for s in range(3)]
    ax.scatter(np.full(3, i) - 0.1, sm, color=C_SCORE, s=12, marker="s", label="Σ_meas (count noise)" if i == 0 else None, zorder=3)
    ax.scatter(np.full(3, i) + 0.1, tr, color=C_FLOW, s=12, label="truth D_eff (48 h)" if i == 0 else None, zorder=3)
    V(f"b_sigma_meas_{meth}_min", min(sm)); V(f"b_sigma_meas_{meth}_max", max(sm))
    V(f"b_truth48_{meth}_min", min(tr)); V(f"b_truth48_{meth}_max", max(tr))
ax.set_xticks([0, 1]); ax.set_xticklabels(["PCA-20", "FA-20"]); ax.set_yscale("log"); ax.set_ylabel("median trace / d")
ax.set_ylim(1e-3, 0.3); ax.legend(fontsize=6, loc="lower center", borderaxespad=0.2, handletextpad=0.1); ax.set_xlim(-0.5, 1.5)
panel_letter(ax, "b", x=-0.45)
prov.panel("b", "Measurement-noise covariance vs true 48-h effective diffusion (median trace/d), per seed")

# c — Spearman with truth over committed cells
ax = axs[2]
vs = [("raw_shared", "raw\nshared"), ("raw_perpulse", "raw\nper pulse"), ("smoothed_shared", "smoothed\nshared"), ("smoothed_perpulse", "smoothed\nper pulse")]
for j, (var, lab) in enumerate(vs):
    h = d[(d.estimator == "score_flow") & (d.variant == var) & (d.method == "pca") & (d.quantity == "trace/d")].sort_values("seed")
    ax.scatter(j + np.array([-0.08, 0, 0.08]), h.spearman_committed, color=C_FLOW, s=12, zorder=3)
    V(f"c_spearman_committed_{var}_min", h.spearman_committed.min()); V(f"c_spearman_committed_{var}_max", h.spearman_committed.max())
ax.axhline(0.5, color="black", ls="--", lw=0.7)
ax.set_xticks(range(len(vs))); ax.set_xticklabels([l.replace("\n", " ") for _, l in vs], fontsize=6, rotation=35, ha="right", rotation_mode="anchor")
ax.set_ylabel("Spearman with truth (committed cells)"); ax.set_ylim(-0.1, 0.6)
ax.set_title("score–flow estimate, PCA, 6 h", fontsize=6.5)
panel_letter(ax, "c", x=-0.35)
prov.panel("c", "Score-flow diffusion estimate: Spearman with truth over committed cells per labelling variant (gate >= 0.5)")
save(fig, "FigS1", kind="supplementary")

[save] supplementary/FigS1: 160.9 x 75.7 mm, PNG 3801 x 1788 px @ 600 dpi


(PosixPath('/Users/terooatt/Documents/scJDO2-figures/manuscript/figures/supplementary/FigS1.pdf'),
 PosixPath('/Users/terooatt/Documents/scJDO2-figures/manuscript/figures/supplementary/FigS1.png'))

In [2]:
v = prov.values
prov.legend(f"""
**Supplementary Figure S1 | Diffusion is not identifiable at typical sampling intervals (S10, pre-registered; gate failed).** v3_default,
seeds 0–2. **a**, Median estimate / median truth of trace(D_eff)/d (truth from 32 forward paths per cell, Δt = 48 h for clone and OT
estimators, 6 h for the score–flow estimator). Gate versions (Σ_meas subtracted as written): clone dispersion
{rng_str(v['a_clone_dispersion_literal_pca_ratio_min'], v['a_clone_dispersion_literal_pca_ratio_max'])} (PCA), OT noise
{rng_str(v['a_ot_noise_minus_trace_sigma_meas_over_d_pca_ratio_min'], v['a_ot_noise_minus_trace_sigma_meas_over_d_pca_ratio_max'])};
uncorrected: clone {rng_str(v['a_clone_dispersion_uncorrected_pca_ratio_min'], v['a_clone_dispersion_uncorrected_pca_ratio_max'])}, OT
{rng_str(v['a_ot_noise_uncorrected_pca_ratio_min'], v['a_ot_noise_uncorrected_pca_ratio_max'])} (the OT held-out selection hit the smallest grid value);
score–flow (raw labelling) {rng_str(v['a_score_flow_raw_shared_pca_ratio_min'], v['a_score_flow_raw_shared_pca_ratio_max'])}. Green band: within a factor of 2.
**b**, Count-noise covariance in the embedding (Σ_meas, counts redrawn from the true pools) exceeds the 48-h effective diffusion:
PCA {rng_str(v['b_sigma_meas_pca_min'], v['b_sigma_meas_pca_max'], 3)} vs {rng_str(v['b_truth48_pca_min'], v['b_truth48_pca_max'], 3)};
FA {rng_str(v['b_sigma_meas_fa_min'], v['b_sigma_meas_fa_max'], 3)} vs {rng_str(v['b_truth48_fa_min'], v['b_truth48_fa_max'], 4)}.
**c**, Spearman between the score–flow estimate and the truth over committed cells (gate ≥ 0.5): raw shared γ
{rng_str(v['c_spearman_committed_raw_shared_min'], v['c_spearman_committed_raw_shared_max'], 3)}; smoothed shared γ
{rng_str(v['c_spearman_committed_smoothed_shared_min'], v['c_spearman_committed_smoothed_shared_max'], 3)}.
""")
prov.write()